In [1]:
pip install ucimlrepo

**Load datasets**

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from google.colab import drive
drive.mount('/content/drive')

# Fetch dataset
bcwd = fetch_ucirepo(id=17)

# Extract features and target
X_bcwd = bcwd.data.features
y_bcwd = bcwd.data.targets

# Convert target to 0 and 1
label_encoder_bcwd = LabelEncoder()
y_bcwd = label_encoder_bcwd.fit_transform(y_bcwd).ravel()

# Ensure data is in pandas DataFrame format
if not isinstance(X_bcwd, pd.DataFrame):
    X_bcwd = pd.DataFrame(X_bcwd)
if not isinstance(y_bcwd, pd.Series):
    if isinstance(y_bcwd, pd.DataFrame):
        y_bcwd = y_bcwd.iloc[:, 0]  # Extract the first column if y is a DataFrame
    else:
        y_bcwd = pd.Series(y_bcwd)

# Combine features and target into one DataFrame
bcwd = pd.concat([X_bcwd, y_bcwd.rename("target")], axis=1)





# Fetch dataset
bcwo = fetch_ucirepo(id=15)

# Extract features and target
X_bcwo = bcwo.data.features
y_bcwo = bcwo.data.targets


label_encoder_bcwo = LabelEncoder()
y_bcwo = label_encoder_bcwo.fit_transform(y_bcwo).ravel()

# Ensure data is in pandas DataFrame format
if not isinstance(X_bcwo, pd.DataFrame):
    X_bcwo = pd.DataFrame(X_bcwo)
if not isinstance(y_bcwo, pd.Series):
    if isinstance(y_bcwo, pd.DataFrame):
        y_bcwo = y_bcwo.iloc[:, 0]  # Extract the first column if y is a DataFrame
    else:
        y_bcwo = pd.Series(y_bcwo)

# Combine features and target into one DataFrame
bcwo = pd.concat([X_bcwo, y_bcwo.rename("target")], axis=1)




file_path = '/content/bcancerfeat.csv'
bct= pd.read_csv(file_path)

bct = bct[bct['target'] != 3]

X_bct = bct.drop(columns=['target'])

y_bct = bct['target']

label_encoder_bct = LabelEncoder()

y_bct = label_encoder_bct.fit_transform(y_bct)

bct['target'] = y_bct


Mounted at /content/drive


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


**Adacost**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import make_scorer, recall_score, confusion_matrix, accuracy_score, f1_score
from scipy.stats import randint, uniform
import warnings

warnings.filterwarnings('ignore')

# Define custom G-Mean, specificity, and other scorers
def specificity_score(y_true, y_pred):
    """
    Calculate the specificity (true negative rate) of the predictions.

    Parameters:
    -----------
    y_true : array-like of shape (n_samples,)
        True binary labels.

    y_pred : array-like of shape (n_samples,)
        Predicted binary labels.

    Returns:
    --------
    specificity : float
        The specificity score, defined as TN / (TN + FP).
    """
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]
    fp = cm[0, 1]
    return tn / (tn + fp)

def geometric_mean_score(y_true, y_pred):
    """
    Calculate the geometric mean of sensitivity (recall) and specificity.

    Parameters:
    -----------
    y_true : array-like of shape (n_samples,)
        True binary labels.

    y_pred : array-like of shape (n_samples,)
        Predicted binary labels.

    Returns:
    --------
    g_mean : float
        The geometric mean score, defined as sqrt(sensitivity * specificity).
    """
    sensitivity = recall_score(y_true, y_pred)
    specificity = specificity_score(y_true, y_pred)
    return np.sqrt(sensitivity * specificity)

# Create the scorers
g_mean_scorer = make_scorer(geometric_mean_score, greater_is_better=True)
specificity = make_scorer(specificity_score)
sensitivity = make_scorer(recall_score)
accuracy = make_scorer(accuracy_score)
f1 = make_scorer(f1_score)

# Custom AdaCost implementation
class AdaCost(BaseEstimator, ClassifierMixin):
    """
    Custom AdaCost classifier for handling class imbalance using a modified AdaBoost approach.

    AdaCost applies different penalties for misclassifications of positive and negative classes,
    making it more sensitive to class imbalance in datasets.

    Parameters:
    -----------
    base_estimator : estimator object, default=None
        The base estimator from which the boosted ensemble is built. If None, a DecisionTreeClassifier
        with max_depth=1 is used.

    n_estimators : int, default=50
        The number of boosting rounds (iterations).

    learning_rate : float, default=1.0
        Weight applied to each estimator at each boosting step. A lower value increases the influence
        of each individual estimator.

    cost_positive : float, default=1.0
        Penalty for misclassifying a positive instance.

    cost_negative : float, default=1.0
        Penalty for misclassifying a negative instance.
    """

    def __init__(self, base_estimator=None, n_estimators=50, learning_rate=1.0, cost_positive=1.0, cost_negative=1.0):
        self.base_estimator = base_estimator if base_estimator is not None else DecisionTreeClassifier(max_depth=1)
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.cost_positive = cost_positive
        self.cost_negative = cost_negative

    def fit(self, X, y):
        """
        Fit the AdaCost model on the provided dataset.

        Parameters:
        -----------
        X : array-like or DataFrame of shape (n_samples, n_features)
            The input feature set.

        y : array-like of shape (n_samples,)
            The target labels.

        Returns:
        --------
        self : object
            Fitted AdaCost model.
        """
        X, y = check_X_y(X, y)
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        self.estimators_ = []
        self.estimator_weights_ = np.zeros(self.n_estimators, dtype=np.float64)

        # Initial weight distribution D(i)
        sample_weight = np.ones(n_samples)
        sample_weight[y == 1] = self.cost_positive / (self.cost_positive + self.cost_negative)
        sample_weight[y == -1] = self.cost_negative / (self.cost_positive + self.cost_negative)
        sample_weight /= np.sum(sample_weight)  # Normalize

        for t in range(self.n_estimators):
            estimator = clone(self.base_estimator)
            estimator.fit(X, y, sample_weight=sample_weight)
            y_pred = estimator.predict(X)

            incorrect = (y_pred != y)
            error = np.dot(sample_weight, incorrect)

            # Prevent division by zero and stop if the error is too high
            if error > 0.5:
                break

            # Calculate alpha_t (goodness of the classifier)
            alpha_t = 0.5 * np.log((1 - error) / max(error, 1e-10))

            # Compute the cost-adjustment function beta(i)
            beta = np.ones_like(y, dtype=np.float64)
            beta[incorrect & (y == 1)] = 0.5 * (1 - self.cost_positive)
            beta[~incorrect & (y == 1)] = 0.5 * (1 + self.cost_positive)
            beta[incorrect & (y == -1)] = 0.5 * (1 - self.cost_negative)
            beta[~incorrect & (y == -1)] = 0.5 * (1 + self.cost_negative)

            # Update weights
            sample_weight *= np.exp(-alpha_t * y * y_pred * beta)
            sample_weight /= np.sum(sample_weight)  # Normalize

            # Save the current estimator and its weight
            self.estimators_.append(estimator)
            self.estimator_weights_[t] = alpha_t

        return self

    def predict(self, X):
        """
        Predict class labels for the input dataset X.

        Parameters:
        -----------
        X : array-like or DataFrame of shape (n_samples, n_features)
            The input feature set.

        Returns:
        --------
        predictions : array-like of shape (n_samples,)
            The predicted class labels.
        """
        check_is_fitted(self, ['estimators_', 'estimator_weights_'])
        X = check_array(X)
        weighted_sum = sum(estimator.predict(X) * weight for estimator, weight in zip(self.estimators_, self.estimator_weights_))
        return np.sign(weighted_sum)

    def score(self, X, y):
        """
        Returns the accuracy of the model on the provided dataset.

        Parameters:
        -----------
        X : array-like or DataFrame of shape (n_samples, n_features)
            The input feature set.

        y : array-like of shape (n_samples,)
            The target labels.

        Returns:
        --------
        score : float
            Accuracy of the model on the provided dataset.
        """
        return accuracy_score(y, self.predict(X))

# Define a pipeline with and without RFE (feature selection) and AdaCost
def build_and_evaluate_model(X, y, dataset_name, n_features_values, use_feature_selection=True):
    """
    Builds, evaluates, and returns the AdaCost model with or without feature selection.

    Parameters:
    -----------
    X : array-like or DataFrame of shape (n_samples, n_features)
        The input feature set.

    y : array-like of shape (n_samples,)
        The target labels.

    dataset_name : str
        Name of the dataset being processed.

    n_features_values : dict
        Dictionary specifying the range of features to select during RFE for each dataset.

    use_feature_selection : bool, default=True
        Boolean indicating whether to use RFE for feature selection.

    Returns:
    --------
    cv_results : dict
        Cross-validation results for various metrics.

    best_params : dict
        Best hyperparameters found during RandomizedSearchCV.

    best_model : estimator object
        Best estimator found during RandomizedSearchCV.

    n_features_selected : int
        Number of features selected (if RFE is used).
    """

    scaler = StandardScaler()


    base_estimator = DecisionTreeClassifier(max_depth=3)

    # Define pipeline components
    steps = [('scaler', scaler)]

    if use_feature_selection:
        rfe = RFE(estimator=base_estimator)
        steps.append(('rfe', rfe))

    adacost = AdaCost(base_estimator=base_estimator, n_estimators=50, learning_rate=0.5, cost_positive=1.5, cost_negative=1.0)
    steps.append(('adacost', adacost))

    pipeline = Pipeline(steps)

    param_space = {
        'adacost__n_estimators': randint(30, 100),
        'adacost__learning_rate': uniform(0.01, 2.0),
        'adacost__cost_positive': uniform(1.0, 5.0),
        'adacost__cost_negative': [1.0]
    }

    if use_feature_selection:
        param_space['rfe__n_features_to_select'] = n_features_values[dataset_name]

    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=101)

    # Randomized Search for hyperparameter tuning with G-Mean as the scoring metric
    search = RandomizedSearchCV(pipeline, param_space, n_iter=30, scoring=g_mean_scorer, cv=stratified_kfold, n_jobs=2, random_state=101)
    search.fit(X, y)

    best_model = search.best_estimator_
    best_params = search.best_params_

    # Extract the number of features selected by RFE if used
    n_features_selected = best_model.named_steps['rfe'].n_features_ if use_feature_selection else X.shape[1]

    cv_results = cross_validate(best_model, X, y, cv=stratified_kfold, scoring={
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'g_mean': g_mean_scorer,
        'f1': f1
    }, return_train_score=False)

    return cv_results, best_params, best_model, n_features_selected

# Ensure features and target are in pandas DataFrame format
def ensure_dataframe(X, y):
    """
    Ensures that the input features and target labels are in pandas DataFrame and Series format respectively.

    Parameters:
    -----------
    X : array-like or DataFrame of shape (n_samples, n_features)
        The input feature set.

    y : array-like, Series, or DataFrame of shape (n_samples,)
        The target labels.

    Returns:
    --------
    X : DataFrame of shape (n_samples, n_features)
        The input features as a DataFrame.

    y : Series of shape (n_samples,)
        The target labels as a Series.
    """
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    if not isinstance(y, pd.Series):
        if isinstance(y, pd.DataFrame):
            y = y.iloc[:, 0]  # Extract the first column if y is a DataFrame
        else:
            y = pd.Series(y)
    return X, y


X_bcwd, y_bcwd = ensure_dataframe(X_bcwd, y_bcwd)
X_bcwo, y_bcwo = ensure_dataframe(X_bcwo, y_bcwo)

# Impute missing values
imputer = KNNImputer(n_neighbors=5)
X_bcwo = pd.DataFrame(imputer.fit_transform(X_bcwo), columns=X_bcwo.columns)


n_features_values = {
    'BCWD': randint(10, 30),
    'BCWO': randint(3, 9),
    'BCT': randint(10, 38)
}

# Define different datasets to test
datasets = {
    'BCWD': (X_bcwd, y_bcwd),
    'BCWO': (X_bcwo, y_bcwo),
    'BCT': (X_bct, y_bct)
}

results = []
best_models = []

for dataset_name, (X, y) in datasets.items():
    X = StandardScaler().fit_transform(X)

    # Evaluate with feature selection
    cv_results_fs, best_params_fs, best_model_fs, n_features_selected_fs = build_and_evaluate_model(X, y, dataset_name, n_features_values, use_feature_selection=True)

    # Store the best results with feature selection
    result_fs = {
        'Dataset': dataset_name,
        'Method': 'With Feature Selection',
        'Accuracy': np.mean(cv_results_fs['test_accuracy']),
        'Sensitivity': np.mean(cv_results_fs['test_sensitivity']),
        'Specificity': np.mean(cv_results_fs['test_specificity']),
        'G-Mean': np.mean(cv_results_fs['test_g_mean']),
        'F1': np.mean(cv_results_fs['test_f1']),
        'Best Params': best_params_fs,
        'Num Features Selected': n_features_selected_fs
    }
    results.append(result_fs)
    best_models.append(best_model_fs)

    # Evaluate without feature selection
    cv_results_nofs, best_params_nofs, best_model_nofs, n_features_selected_nofs = build_and_evaluate_model(X, y, dataset_name, n_features_values, use_feature_selection=False)

    # Store the best results without feature selection
    result_nofs = {
        'Dataset': dataset_name,
        'Method': 'Without Feature Selection',
        'Accuracy': np.mean(cv_results_nofs['test_accuracy']),
        'Sensitivity': np.mean(cv_results_nofs['test_sensitivity']),
        'Specificity': np.mean(cv_results_nofs['test_specificity']),
        'G-Mean': np.mean(cv_results_nofs['test_g_mean']),
        'F1': np.mean(cv_results_nofs['test_f1']),
        'Best Params': best_params_nofs,
        'Num Features Selected': n_features_selected_nofs
    }
    results.append(result_nofs)
    best_models.append(best_model_nofs)

results_adacost = pd.DataFrame(results).round(4)
results_adacost.to_csv('results_adacost_with_and_without_fs.csv', index=False)

print(results_adacost.round(4))


  Dataset                     Method  Accuracy  Sensitivity  Specificity  \
0    BCWD     With Feature Selection    0.9157       0.9619       0.8883   
1    BCWD  Without Feature Selection    0.8946       0.9576       0.8572   
2    BCWO     With Feature Selection    0.9671       0.9628       0.9694   
3    BCWO  Without Feature Selection    0.9585       0.9710       0.9519   
4     BCT     With Feature Selection    0.5952       0.6667       0.5795   
5     BCT  Without Feature Selection    0.5838       0.6167       0.5758   

   G-Mean      F1                                        Best Params  \
0  0.9236  0.8956  {'adacost__cost_negative': 1.0, 'adacost__cost...   
1  0.9048  0.8741  {'adacost__cost_negative': 1.0, 'adacost__cost...   
2  0.9659  0.9529  {'adacost__cost_negative': 1.0, 'adacost__cost...   
3  0.9611  0.9424  {'adacost__cost_negative': 1.0, 'adacost__cost...   
4  0.5715  0.3785  {'adacost__cost_negative': 1.0, 'adacost__cost...   
5  0.5515  0.3618  {'adacost__cost_